In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import ParameterGrid

import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

from itertools import product
import joblib
from datetime import datetime
from pathlib import Path
from tqdm.auto import tqdm
import os, sys, logging, gc

sys.path.append('../../')
import src.forecasting.simulations      as sim
import src.fda.kde.estimators           as kde
import src.fda.transformations.lqdt     as lqdt
import src.forecasting.cross_validation as cv
import src.forecasting.pipelines        as fp
import src.forecasting.accuracy         as acc
import src.fda.utils                    as fdaUtils

In [2]:
EXECUTION_DATE : str = datetime.now().strftime('%Y%m%d')
# EXECUTION_DATE = '20260621'

LOG_PATH        : str = f'../../logs/simulations/{EXECUTION_DATE}.log'
SIM_MAIN_PATH   : str = f'../../data/interim/simulation/{EXECUTION_DATE}/'
SIMS_PATH       : str = f'../../data/interim/simulation/{EXECUTION_DATE}/00_simulations/'
SIMS_LQD_PATH   : str = f'../../data/interim/simulation/{EXECUTION_DATE}/01_lqds/'
KDES_PATH       : str = f'../../data/interim/simulation/{EXECUTION_DATE}/02_kdes/'
KDES_LQDS_PATH  : str = f'../../data/interim/simulation/{EXECUTION_DATE}/03_kde_lqds/'
CV_FCS_PATH     : str = f'../../data/interim/simulation/{EXECUTION_DATE}/04_cv/'

Path(SIM_MAIN_PATH).mkdir(parents=True, exist_ok=True)
Path(SIMS_PATH).mkdir(parents=True, exist_ok=True)
Path(SIMS_LQD_PATH).mkdir(parents=True, exist_ok=True)
Path(KDES_PATH).mkdir(parents=True, exist_ok=True)
Path(KDES_LQDS_PATH).mkdir(parents=True, exist_ok=True)
Path(CV_FCS_PATH).mkdir(parents=True, exist_ok=True)
Path(LOG_PATH).parent.mkdir(parents=True, exist_ok=True)

# GAS model

In [3]:
# Define scenarios
gas_params = {
    # "scenario_1": {
    #     "Description": "Location-driven ($m_t$)",
    #     "alpha": np.diag([0.08, 0.01, 0.01]),
    #     "beta": np.diag([0.90, 0.95, 0.95]),
    # },
    # "scenario_2": {
    #     "Description": "Scale-driven ($\\sigma_t$)",
    #     "alpha": np.diag([0.01, 0.08, 0.01]),
    #     "beta": np.diag([0.95, 0.90, 0.95]),
    # },
    # "scenario_3": {
    #     "Description": "Shape-driven ($\\eta_t$)",
    #      "alpha": np.diag([0.01, 0.01, 0.08]),
    #       "beta": np.diag([0.95, 0.95, 0.90])
    # },
    "scenario_4": {
        "Description": "Mixture",
        "alpha": np.diag([0.04, 0.06, 0.04]),
        "beta": np.diag([0.92, 0.95, 0.94]),
    }
}

distribution_params = {
    "nu": [3, 8]
}

In [4]:
keys = ["scenario", "nu"]
values = [list(gas_params.keys()), distribution_params["nu"]]

param_grid = []

for scenario, nu in product(*values):
    gas_cfg = gas_params[scenario]

    param_grid.append({
        "scenario": "___nu=".join([scenario, str(nu)]),
        "nu": nu,
        "alpha": gas_cfg["alpha"],
        "beta": gas_cfg["beta"]    
})

In [5]:
# grid for densities
x = np.linspace(-40, 40, 5001)
# number of curves (densities)
T = 301
# number of simulations (f_{N_REP,1},...,f_{N_REP,T})
N_REPS = 1_000
# number of samples from each f_t density
N_SAMPLES = 288

HORIZON = 1
CV_WINDOW_TYPE = "expanding"
CV_WINDOW_SIZE = T - HORIZON

SIMULATION_END_DATE = pd.Timestamp.today().normalize()
total = len(param_grid) * N_REPS

In [6]:
total = len(param_grid) * N_REPS
pbar = tqdm(total=total, desc="Total Simulation Progress")

sim_addresses = []

for params in param_grid:
    scenario = params["scenario"]

    for n_rep in range(N_REPS):
        sim_path = f"{SIMS_PATH}{scenario}___rep_{n_rep}.jbl"

        if Path(sim_path).exists():
            sim_addresses.append({
                "scenario": scenario,
                "n_rep": n_rep,
                "address": sim_path
            })
            pbar.update(1)
            continue

        # 1. Setup Model and Simulate
        gm = sim.GasModel(alpha=params["alpha"], beta=params["beta"], nu=params["nu"])
        sim_results = gm.simulate(T=T, burn_in=300)

        # 2. Get Theoretical Conditional Densities
        sim_density = gm.conditional_densities(grid=x, theta_path=sim_results["theta"])
        dates = pd.date_range(end=SIMULATION_END_DATE, periods=T, freq="D")
        sim_density.columns = dates

        # 3. Generate Samples Efficiently
        samples_list = []
        for i in range(len(sim_results["theta"])):
            sample = gm.rvs(n=N_SAMPLES, theta=sim_results["theta"][i])
            samples_list.append(sample)

        df_samples = pd.DataFrame(np.array(samples_list).T, columns=dates)

        rep_result = {
            "scenario": scenario,
            "n_rep": n_rep,
            "params": params,
            "theta": sim_results["theta"],
            "densities": sim_density,
            "samples": df_samples
        }

        joblib.dump(rep_result, sim_path)
        sim_addresses.append({
            "scenario": scenario,
            "n_rep": n_rep,
            "address": sim_path
        })

        del gm, sim_results, sim_density, samples_list, df_samples, rep_result
        gc.collect()
        pbar.update(1)

joblib.dump(sim_addresses, f"{SIM_MAIN_PATH}simulation_addresses.jbl")

Total Simulation Progress:   0%|          | 0/2000 [00:00<?, ?it/s]

['../../data/interim/simulation/20260728/simulation_addresses.jbl']

In [7]:
sim_jbls_path = SIMS_PATH
sim_paths = [
    str(Path(sim_jbls_path) / x)
    for x in os.listdir(sim_jbls_path)
    if x.endswith(".jbl") and "___rep_" in x
]

sim_addresses = []
for path_name in sim_paths:
    obj = joblib.load(path_name)
    sim_address = {
        "scenario": obj["scenario"],
        "n_rep": obj["n_rep"],
        "address": path_name
    }
    sim_addresses.append(sim_address)
    del obj

sim_addresses = sorted(
    sim_addresses,
    key=lambda x: (x["scenario"], x["n_rep"])
)

sim_index = {
    (d["scenario"], d["n_rep"]): d["address"]
    for d in sim_addresses
}

joblib.dump(sim_addresses, f"{SIM_MAIN_PATH}simulation_addresses.jbl")
print(f"Indexed {len(sim_addresses)} simulation files")

Indexed 2000 simulation files


In [8]:
# # Plotting simulations
# doc_path = "../../densities4risk_doc/Figures/"
# # ==========================================
# # Metadata
# # ==========================================

# nus = distribution_params["nu"]
# scenario_names = list(gas_params.keys())

# # ==========================================
# # Figure
# # ==========================================

# n_rows = len(scenario_names)
# n_cols = len(nus)

# fig, axes = plt.subplots(
#     n_rows,
#     n_cols,
#     figsize=(5 * n_cols, 3.5 * n_rows),
#     dpi=300,
#     constrained_layout=True,
#     squeeze=False
# )

# # ==========================================
# # Plot loop
# # ==========================================

# for i, scenario in enumerate(scenario_names):

#     for j, nu in enumerate(nus):

#         ax = axes[i, j]

#         key = f"{scenario}___nu={nu}"

#         sim_path = sim_index.get((key, 0))
#         if sim_path is None:
#             ax.set_visible(False)
#             continue

#         sim_obj = joblib.load(sim_path)
#         data = sim_obj["densities"]

#         # ----------------------------------
#         # Background densities
#         # ----------------------------------

#         data.plot(
#             color="gray",
#             alpha=0.03,
#             linewidth=0.5,
#             legend=False,
#             ax=ax
#         )

#         ax.set_xlim([-10,10])

#         # ----------------------------------
#         # First curve
#         # ----------------------------------

#         first_line = data.iloc[:, 0].plot(
#             color="#70e000",
#             linewidth=2.2,
#             ax=ax,
#             label="t=1"
#         )

#         # ----------------------------------
#         # Last curve
#         # ----------------------------------

#         last_line = data.iloc[:, -1].plot(
#             color="#e5383b",
#             linewidth=2.2,
#             ax=ax,
#             label="t=T"
#         )

#         # ==================================
#         # Titles / labels
#         # ==================================

#         if i == 0:
#             ax.set_title(
#                 rf"$\nu={nu}$",
#                 fontsize=15,
#                 fontweight="bold",
#                 pad=10
#             )

#         if j == 0:
#             ax.set_ylabel(
#                 gas_params[scenario]["Description"],
#                 # scenario.replace("_", " ").title() + "\nDensity",
#                 fontsize=12
#             )

#         ax.set_xlabel(r"$x$", fontsize=11)

#         # ==================================
#         # Styling
#         # ==================================

#         ax.spines["top"].set_visible(False)
#         ax.spines["right"].set_visible(False)

#         ax.grid(
#             True,
#             linestyle="--",
#             alpha=0.2
#         )

# # ==========================================
# # Shared legend
# # ==========================================

# handles = [
#     axes[0, 0].lines[-2],
#     axes[0, 0].lines[-1]
# ]

# labels = ["t=1", "t=T"]

# fig.legend(
#     handles,
#     labels,
#     loc="upper center",
#     ncol=2,
#     frameon=False,
#     fontsize=11
# )

# # plt.savefig(''.join([doc_path, "sinape_gasSim_all.png"]), bbox_inches='tight') # PDFs are better for LaTeX
# plt.show()

# KDE

In [9]:
rot_grid = {
    "method": ["rot"], #rot/2, --> h elevado reduz as dinâmicas captadas pelo estimador (não considerar 2*ROT)
    "kernel": ["gaussian"],
    "sigma_robust": [False]
}

density_param_grid = {}
for grid in [rot_grid]:
    for params in ParameterGrid(grid):
        
        key_parts = [params["kernel"]]
        if "method" in params: key_parts.append(params["method"])
        if "df" in params: key_parts.append(f"df={params['df']}")
        if params.get("sigma_robust"): key_parts.append("robust")
        if params.get("cv") == "LOO": key_parts.append("loo")
        
        full_model_name = "_".join(key_parts).replace(".", "")
        
        kernel_label = params["kernel"]
        if "df" in params:
            kernel_label += f"+df={params['df']}"
            
        bw_label = params.get("method", "fixed")
        if params.get("sigma_robust"): bw_label += "_robust"
        if params.get("cv") == "LOO": bw_label += "_loo"


        density_param_grid[full_model_name] = {
            "kernel": kernel_label,
            "bandwidth": bw_label,
            "params": params  
        }

print("KDE models:")
for name, values in density_param_grid.items():
    print("\t",name, ":", values["params"])

print(f"Total KDE models: {len(density_param_grid.items())}")

KDE models:
	 gaussian_rot : {'kernel': 'gaussian', 'method': 'rot', 'sigma_robust': False}
Total KDE models: 1


In [10]:
total = len(sim_addresses) * len(density_param_grid)
pbar = tqdm(total=total, desc="Total KDE Progress")

for sim_address in sim_addresses:
    scenario = sim_address["scenario"]
    n_rep = sim_address["n_rep"]
    scenario_sim_dict = joblib.load(sim_address["address"])
    returns_df = scenario_sim_dict["samples"]

    print(f"Processing KDEs for {scenario} | rep {n_rep}")

    for kde_bw_name, kde_bw_params in density_param_grid.items():
        kde_path = f"{KDES_PATH}{scenario}___rep_{n_rep}___kde_{kde_bw_name}.jbl"

        if Path(kde_path).exists():
            pbar.update(1)
            continue

        print(f"	{kde_bw_name}")

        # bandwidths
        df_h = kde.df_bandwidth_selector(returns_df, **kde_bw_params["params"])
        kde_params = {k: v for k, v in kde_bw_params["params"].items() if k in ["kernel", "df"]}

        # KDEs for samples from f_t
        df_grids, df_densities = kde.df_to_kde(
            X=returns_df,
            h=df_h,
            normalize_densities=False,
            **kde_params
        )

        rep_result = {
            "scenario": scenario,
            "n_rep": n_rep,
            "model_name": kde_bw_name,
            "kde_params": kde_bw_params["kernel"],
            "kernel": kde_bw_params["params"]["kernel"],
            "bw_params": kde_bw_params["bandwidth"],
            "bw_method": kde_bw_params["params"]["method"],
            "df_h": df_h,
            "df_support": df_grids,
            "df_densities": df_densities
        }

        joblib.dump(rep_result, kde_path)

        del df_h, df_grids, df_densities, kde_params, rep_result
        gc.collect()
        pbar.update(1)

    del scenario_sim_dict, returns_df
    gc.collect()

Total KDE Progress:   0%|          | 0/2000 [00:00<?, ?it/s]

Processing KDEs for scenario_4___nu=3 | rep 0
	gaussian_rot
Processing KDEs for scenario_4___nu=3 | rep 1
	gaussian_rot
Processing KDEs for scenario_4___nu=3 | rep 2
	gaussian_rot
Processing KDEs for scenario_4___nu=3 | rep 3
	gaussian_rot
Processing KDEs for scenario_4___nu=3 | rep 4
	gaussian_rot
Processing KDEs for scenario_4___nu=3 | rep 5
	gaussian_rot
Processing KDEs for scenario_4___nu=3 | rep 6
	gaussian_rot
Processing KDEs for scenario_4___nu=3 | rep 7
	gaussian_rot
Processing KDEs for scenario_4___nu=3 | rep 8
	gaussian_rot
Processing KDEs for scenario_4___nu=3 | rep 9
	gaussian_rot
Processing KDEs for scenario_4___nu=3 | rep 10
	gaussian_rot
Processing KDEs for scenario_4___nu=3 | rep 11
	gaussian_rot
Processing KDEs for scenario_4___nu=3 | rep 12
	gaussian_rot
Processing KDEs for scenario_4___nu=3 | rep 13
	gaussian_rot
Processing KDEs for scenario_4___nu=3 | rep 14
	gaussian_rot
Processing KDEs for scenario_4___nu=3 | rep 15
	gaussian_rot
Processing KDEs for scenario_4___n

In [11]:
kde_jbls_path = KDES_PATH
kdes_paths = [
    str(Path(kde_jbls_path) / x)
    for x in os.listdir(kde_jbls_path)
    if x.endswith(".jbl") and "___rep_" in x
]

kde_addresses = []
for path_name in kdes_paths:
    obj = joblib.load(path_name)
    kde_address = {
        "scenario":   obj["scenario"],
        "n_rep":      obj["n_rep"],
        "model_name": obj["model_name"],
        "address":    path_name
    }
    kde_addresses.append(kde_address)

kde_addresses = sorted(
    kde_addresses,
    key=lambda x: (x["scenario"], x["n_rep"], x["model_name"])
)

kde_index = {
    (d["scenario"], d["n_rep"], d["model_name"]): d["address"]
    for d in kde_addresses
}

joblib.dump(kde_addresses, f"{SIM_MAIN_PATH}kde_addresses.jbl")
print(f"Indexed {len(kde_addresses)} KDE files")

Indexed 2000 KDE files


# LQD transforms

In [12]:
n_sim_addresses = len(sim_addresses)
pbar = tqdm(total=n_sim_addresses, desc="Total LQD(f conditional) Progress")

for sim_address in sim_addresses:
    scenario = sim_address["scenario"]
    n_rep = sim_address["n_rep"]
    sim_lqd_path = f"{SIMS_LQD_PATH}lqd___scenario_{scenario}___rep_{n_rep}.jbl"

    if Path(sim_lqd_path).exists():
        pbar.update(1)
        continue

    scenario_sim_dict = joblib.load(sim_address["address"])
    sim_densities = scenario_sim_dict["densities"]
    sim_densities_supp = sim_densities.copy()
    sim_densities_supp.loc[:, :] = sim_densities_supp.index.to_numpy()[:, None]

    mlqdt = lqdt.mLQDT()
    model_lqd = mlqdt.transform(
        densities=sim_densities,
        densities_supports=sim_densities_supp,
        verbose=False
    )

    conditional_mlqdt = model_lqd.lqd.copy()
    conditional_mlqdt.index = model_lqd.lqd_support

    sim_mlqdt_result = {
        "scenario": scenario,
        "n_rep": n_rep,
        "conditional_mlqdt": conditional_mlqdt,
        "c": model_lqd.c,
        "t0": model_lqd.t0
    }

    joblib.dump(sim_mlqdt_result, sim_lqd_path)

    del scenario_sim_dict, sim_densities, sim_densities_supp, mlqdt, model_lqd, conditional_mlqdt, sim_mlqdt_result
    gc.collect()
    pbar.update(1)


Total LQD(f conditional) Progress:   0%|          | 0/2000 [00:00<?, ?it/s]

In [13]:
sim_lqd_paths = [
    str(Path(SIMS_LQD_PATH) / x)
    for x in os.listdir(SIMS_LQD_PATH)
    if x.endswith(".jbl") and "___rep_" in x
]

sim_lqd_addresses = []
for path_name in sim_lqd_paths:
    obj = joblib.load(path_name)
    sim_lqd_address = {
        "scenario": obj["scenario"],
        "n_rep": obj["n_rep"],
        "address": path_name
    }
    sim_lqd_addresses.append(sim_lqd_address)
    del obj

sim_lqd_addresses = sorted(
    sim_lqd_addresses,
    key=lambda x: (x["scenario"], x["n_rep"])
)

sim_lqd_index = {
    (d["scenario"], d["n_rep"]): d["address"]
    for d in sim_lqd_addresses
}

joblib.dump(sim_lqd_addresses, f"{SIM_MAIN_PATH}simulation_lqd_addresses.jbl")
print(f"Indexed {len(sim_lqd_addresses)} conditional LQD files")


Indexed 2000 conditional LQD files


In [14]:
n_kde_addresses = len(kde_addresses)
pbar = tqdm(total=n_kde_addresses, desc="Total LQD(fhat) Progress")

for kde_address in kde_addresses:
    scenario_kde_dict = joblib.load(kde_address["address"])
    id_scenario_kde = scenario_kde_dict["scenario"]
    n_rep = scenario_kde_dict["n_rep"]
    model_name_path = scenario_kde_dict["model_name"]

    kde_lqd_path = f"{KDES_LQDS_PATH}lqd___scenario_{id_scenario_kde}___rep_{n_rep}___kde_{model_name_path}.jbl"

    if Path(kde_lqd_path).exists():
        pbar.update(1)
        del scenario_kde_dict
        continue

    mlqdt = lqdt.mLQDT()
    model_lqd = mlqdt.transform(
        densities=scenario_kde_dict["df_densities"],
        densities_supports=scenario_kde_dict["df_support"],
        verbose=False
    )

    Y_t = model_lqd.lqd.copy()
    Y_t.index = model_lqd.lqd_support

    kde_mlqdt_result = {
        "scenario": kde_address["scenario"],
        "n_rep": kde_address["n_rep"],
        "model_name": kde_address["model_name"],
        "kde_mlqdt": Y_t
    }

    joblib.dump(kde_mlqdt_result, kde_lqd_path)

    del scenario_kde_dict, mlqdt, model_lqd, Y_t, kde_mlqdt_result
    gc.collect()
    pbar.update(1)


Total LQD(fhat) Progress:   0%|          | 0/2000 [00:00<?, ?it/s]

/opt/anaconda3/envs/densities4risk/lib/python3.11/site-packages/scipy/interpolate/_interpolate.py:501: RuntimeWarning: overflow encountered in divide
  slope = (y_hi - y_lo) / (x_hi - x_lo)[:, None]
/opt/anaconda3/envs/densities4risk/lib/python3.11/site-packages/scipy/interpolate/_interpolate.py:504: RuntimeWarning: invalid value encountered in multiply
  y_new = slope*(x_new - x_lo)[:, None] + y_lo
/opt/anaconda3/envs/densities4risk/lib/python3.11/site-packages/scipy/interpolate/_interpolate.py:501: RuntimeWarning: divide by zero encountered in divide
  slope = (y_hi - y_lo) / (x_hi - x_lo)[:, None]


In [15]:
kdes_lqd_paths = [
    str(Path(KDES_LQDS_PATH) / x)
    for x in os.listdir(KDES_LQDS_PATH)
    if x.endswith(".jbl") and "___rep_" in x
]

kdes_lqd_addresses = []
for path_name in kdes_lqd_paths:
    obj = joblib.load(path_name)
    kde_address = {
        "scenario": obj["scenario"],
        "n_rep": obj["n_rep"],
        "model_name": obj["model_name"],
        "address": path_name
    }
    kdes_lqd_addresses.append(kde_address)
    del obj

kdes_lqd_addresses = sorted(
    kdes_lqd_addresses,
    key=lambda x: (x["scenario"], x["n_rep"], x["model_name"])
)

kde_lqd_index = {
    (d["scenario"], d["n_rep"], d["model_name"]): d["address"]
    for d in kdes_lqd_addresses
}

joblib.dump(kdes_lqd_addresses, f"{SIM_MAIN_PATH}kde_lqd_addresses.jbl")
print(f"Indexed {len(kdes_lqd_addresses)} KDE LQD files")


Indexed 2000 KDE LQD files


# Cross-validation

In [16]:
logger = logging.getLogger("Density_Estimation")
logger.setLevel(logging.DEBUG)

log_file = str(Path(LOG_PATH).resolve())
if not any(isinstance(handler, logging.FileHandler) and handler.baseFilename == log_file for handler in logger.handlers):
    file_handler = logging.FileHandler(LOG_PATH)
    file_handler.setFormatter(logging.Formatter('%(asctime)s | %(levelname)s | %(message)s'))
    logger.addHandler(file_handler)

logging.captureWarnings(True)


In [17]:
KdFPC_kwargs = {
    "p": 5,
    "dimension": 3
}

In [18]:
def append_curve_comparison(
    measures,
    *,
    scenario,
    n_rep,
    model_name,
    fold,
    comparison,
    fc_date,
    test,
    forecast,
    support=None,
    **kwargs  # <-- Add this to capture any extra arguments
):
    acc_measures = acc.overall_measures(forecast=forecast, test=test)
    support_values = test.index if support is None else support

    temp_df = pd.DataFrame({
        "support": support_values,
        "actual": test.iloc[:, 0].values,
        "forecast": forecast.iloc[:, 0].values,
        "residuals": forecast.iloc[:, 0].values - test.iloc[:, 0].values,
        "date": fc_date,
        "fold": fold
    }).set_index(["fold", "date", "support"])

    measures.append({
        "scenario": scenario,
        "n_rep": n_rep,
        "model_name": model_name,
        "fold": fold,
        "comparison": comparison,
        "fc_date": fc_date,
        **acc_measures,
        **kwargs,  # <-- Unpack the extra arguments here
        "curves": temp_df
    })

In [19]:
total_models = len(kde_addresses)
pbar = tqdm(total=total_models, desc="Total CV Progress")

logger.info("\nInitiating cross validation...")
for kde_address in kde_addresses:
    print(kde_address)
    scenario_kde_dict = joblib.load(kde_address["address"])

    # KDE database
    id_scenario_kde = scenario_kde_dict["scenario"]
    n_rep = scenario_kde_dict["n_rep"]
    model_name_path = scenario_kde_dict["model_name"]
    model_name = scenario_kde_dict["model_name"].replace("_", " ")

    # Simulation database, loaded only for this scenario/replication
    sim_address = sim_index.get((id_scenario_kde, n_rep))
    if sim_address is None:
        raise FileNotFoundError(f"Missing simulation address for {(id_scenario_kde, n_rep)}")

    scenario_sim_dict = joblib.load(sim_address)
    sim_info = scenario_sim_dict["params"]

    logger.info(
        f"Scenario: {id_scenario_kde} | n_rep: {n_rep} | KDE model: {model_name}"
    )

    df_support, df_densities = scenario_kde_dict["df_support"], scenario_kde_dict["df_densities"]

    # Conditional simulated densities f_t
    sim_densities = scenario_sim_dict["densities"]
    returns_df = scenario_sim_dict["samples"]
    sim_densities_supp = sim_densities.copy()
    sim_densities_supp.loc[:, :] = sim_densities_supp.index.to_numpy()[:, None]

    # LQD of conditional densities T(f_t)
    sim_lqd_address = sim_lqd_index.get((id_scenario_kde, n_rep))
    if sim_lqd_address is None:
        raise FileNotFoundError(f"Missing conditional LQD address for {(id_scenario_kde, n_rep)}")
    scenario_sim_lqd = joblib.load(sim_lqd_address)["conditional_mlqdt"]
    # LQD of KDE densities T(fhat_t)
    kde_lqd_address = kde_lqd_index.get((id_scenario_kde, n_rep, scenario_kde_dict["model_name"]))
    if kde_lqd_address is None:
        raise FileNotFoundError(f"Missing KDE LQD address for {(id_scenario_kde, n_rep, scenario_kde_dict['model_name'])}")
    scenario_mlqdt = joblib.load(kde_lqd_address)
    scenario_kde_lqd = scenario_mlqdt["kde_mlqdt"]
    windows = cv.cv_window(T=df_densities.shape[1], h=HORIZON, window_type=CV_WINDOW_TYPE, window_size=CV_WINDOW_SIZE)
    measures = []

    for fold, window in enumerate(windows):
        fold += 1
        print(f"		>>> cv {fold}/{len(windows)}")
        idx_train = window[0]
        idx_test = window[1]
        test_date = df_densities.columns[idx_test]
        fc_date = test_date[0]

        # Targets
        Y_cond_t = scenario_sim_lqd.loc[:, test_date]
        Y_kde_t = scenario_kde_lqd.loc[:, test_date]
        f_hat_t_supp, f_hat_t = df_support.loc[:, test_date], df_densities.loc[:, test_date]
        f_t_support, f_t = sim_densities_supp.loc[:, test_date], sim_densities.loc[:, test_date]

        # Train-test split from KDE densities
        kde_train_support, kde_train = df_support.iloc[:, idx_train], df_densities.iloc[:, idx_train]
        returns_train = returns_df.iloc[:, idx_train]

        # =========================================================================
        # 1. Functional Principal Component (FPC) Loops
        # =========================================================================
        # Forecasting fhat_{t+1}; returns both T-space and inverse-density forecasts
        ## Iterate over FPC approaches
        # for fpc_style in ["dynamic", "static"]:
        for fpc_style in ["dynamic"]:#, "wavelet"]:
            for regression_model in ["randomforest", "adalasso"]:
                for fpc_dimension in [3,10]:
                    print(f"		    >>> {fpc_style}, {regression_model}, {fpc_dimension}")

                    KdFPC_kwargs.update({'dimension': fpc_dimension})
                    forecaster = fp.SimulationForecaster(
                        kdfpc_kwargs=KdFPC_kwargs.copy(),
                        forecaster_model=regression_model
                    )

                    forecaster.fit(
                        kde_train,
                        kde_train_support,
                        returns=returns_train,
                        fpc_style=fpc_style
                    )

                    sim_fc = forecaster.predict(
                        horizon=HORIZON,
                        forecast_index=test_date
                    )

                    Y_hat_t = sim_fc["future_L2_curves"]

                    # support in LQD space
                    Y_hat_t_support = Y_hat_t.copy()
                    Y_hat_t_support.iloc[:, :] = (
                        forecaster.model_lqd.lqd_support[:, None]
                    )

                    current_model_name = f"{model_name} - {fpc_style}"

                    # T(f_kde forecast) x T(f_kde)
                    append_curve_comparison(
                        measures,
                        scenario=id_scenario_kde,
                        n_rep=n_rep,
                        model_name=current_model_name,
                        fold=fold,
                        comparison="TfHatFc_TfHat",
                        fc_date=fc_date,
                        test=Y_kde_t,
                        forecast=Y_hat_t,
                        fpc_style = fpc_style,
                        regression_model = regression_model,
                        ast_kde_samples = 0,
                        fpc_dimension = fpc_dimension
                    )

                    # T(f_kde forecast) x T(f_conditional)
                    append_curve_comparison(
                        measures,
                        scenario=id_scenario_kde,
                        n_rep=n_rep,
                        model_name=current_model_name,
                        fold=fold,
                        comparison="TfHatFc_Tf",
                        fc_date=fc_date,
                        test=Y_cond_t,
                        forecast=Y_hat_t,
                        fpc_style = fpc_style,
                        regression_model = regression_model,
                        ast_kde_samples = 0,
                        fpc_dimension = fpc_dimension
                    )

        # =========================================================================
        # 2. Parametric Benchmark Model (New Integration)
        # =========================================================================
        print(f"            >>> Parametric AST Benchmark")
        
        for nu in [3,6]:
            print(f"                  >>> nu = {nu}")
            # Instantiate and fit using fold training returns
            parametric_model = fp.ParametricDensityForecaster(nu=nu, maxlags=10, criteria="bic")
            parametric_model.fit(returns_train)
            
            # Pulling the spatial grid support from the active test fold sequence
            test_support_vector = f_hat_t_supp.iloc[:, 0]

            for ast_fc_method in ["density", "kde"]:
                print(f"                        >>> forecast object = {ast_fc_method}")
                if ast_fc_method == "kde":
                    for n_kde_forecasts in [1, 1000]:
                            print(f"                                >>> N samples for KDE = {n_kde_forecasts}")
                            parametric_fc = parametric_model.predict(
                                horizon=HORIZON,
                                support=test_support_vector,
                                forecast_index=test_date,
                                forecast=ast_fc_method,
                                n_kde_forecasts = n_kde_forecasts
                            )
                            Y_hat_t_param = parametric_fc["future_L2_curves"]
                            parametric_model_label = "Parametric AST"

                            # T(f_parametric forecast) x T(f_kde)
                            append_curve_comparison(
                                measures,
                                scenario=id_scenario_kde,
                                n_rep=n_rep,
                                model_name=parametric_model_label,
                                fold=fold,
                                comparison="TfHatFc_TfHat",
                                fc_date=fc_date,
                                test=Y_kde_t,
                                forecast=Y_hat_t_param,
                                fpc_style="parametric",
                                regression_model=ast_fc_method,
                                ast_kde_samples=n_kde_forecasts,
                                fpc_dimension=nu
                            )

                            # T(f_parametric forecast) x T(f_conditional)
                            append_curve_comparison(
                                measures,
                                scenario=id_scenario_kde,
                                n_rep=n_rep,
                                model_name=parametric_model_label,
                                fold=fold,
                                comparison="TfHatFc_Tf",
                                fc_date=fc_date,
                                test=Y_cond_t,
                                forecast=Y_hat_t_param,
                                fpc_style="parametric",
                                regression_model=ast_fc_method,
                                ast_kde_samples=n_kde_forecasts,
                                fpc_dimension=nu
                            )
                else:
                    parametric_fc = parametric_model.predict(
                        horizon=HORIZON,
                        support=test_support_vector,
                        forecast_index=test_date,
                        forecast=ast_fc_method
                    )
            
                    Y_hat_t_param = parametric_fc["future_L2_curves"]
                    parametric_model_label = "Parametric AST"

                    # T(f_parametric forecast) x T(f_kde)
                    append_curve_comparison(
                        measures,
                        scenario=id_scenario_kde,
                        n_rep=n_rep,
                        model_name=parametric_model_label,
                        fold=fold,
                        comparison="TfHatFc_TfHat",
                        fc_date=fc_date,
                        test=Y_kde_t,
                        forecast=Y_hat_t_param,
                        fpc_style="parametric",
                        regression_model=ast_fc_method,
                        ast_kde_samples = 0,
                        fpc_dimension=nu
                    )

                    # T(f_parametric forecast) x T(f_conditional)
                    append_curve_comparison(
                        measures,
                        scenario=id_scenario_kde,
                        n_rep=n_rep,
                        model_name=parametric_model_label,
                        fold=fold,
                        comparison="TfHatFc_Tf",
                        fc_date=fc_date,
                        test=Y_cond_t,
                        forecast=Y_hat_t_param,
                        fpc_style="parametric",
                        regression_model=ast_fc_method,
                        ast_kde_samples = 0,
                        fpc_dimension=nu
                    )

    joblib.dump(measures, f"{CV_FCS_PATH}cvFc___scenario_{id_scenario_kde}___rep_{n_rep}___kde_{model_name_path}.jbl")

    del scenario_kde_dict, scenario_sim_dict, df_support, df_densities, sim_densities, sim_densities_supp
    del scenario_sim_lqd, scenario_mlqdt, scenario_kde_lqd, measures
    gc.collect()
    pbar.update(1)

Total CV Progress:   0%|          | 0/2000 [00:00<?, ?it/s]

{'scenario': 'scenario_4___nu=3', 'n_rep': 0, 'model_name': 'gaussian_rot', 'address': '../../data/interim/simulation/20260728/02_kdes/scenario_4___nu=3___rep_0___kde_gaussian_rot.jbl'}
		>>> cv 1/1
		    >>> dynamic, randomforest, 3
		    >>> dynamic, randomforest, 10
		    >>> dynamic, adalasso, 3
		    >>> dynamic, adalasso, 10
            >>> Parametric AST Benchmark
                  >>> nu = 3
                        >>> forecast object = density
                        >>> forecast object = kde
                                >>> N samples for KDE = 1
                                >>> N samples for KDE = 1000
                  >>> nu = 6
                        >>> forecast object = density
                        >>> forecast object = kde
                                >>> N samples for KDE = 1
                                >>> N samples for KDE = 1000
{'scenario': 'scenario_4___nu=3', 'n_rep': 1, 'model_name': 'gaussian_rot', 'address': '../../data/interim/simulation/2026

# Results

In [31]:
cv_results = []
cv_files = [str(Path(CV_FCS_PATH) / x) for x in os.listdir(CV_FCS_PATH) if x.endswith(".jbl")]
# for file in cv_files:
#     cv_results.append(pd.DataFrame(joblib.load(file)).head(10))
# df_cv_results = pd.concat(cv_results)
# df_cv_results.sort_values(by=["scenario", "n_rep", "model_name"], inplace=True)
# df_cv_results.reset_index(inplace=True, drop=True)

In [21]:
from collections import defaultdict
import numpy as np
import pandas as pd
import joblib

acc = defaultdict(lambda: [0.0, 0.0, 0])

for file in cv_files:

    df = pd.DataFrame(joblib.load(file))

    for row in df.itertuples(index=False):

        curves = (
            row.curves
            .reset_index()
            .iloc[1:-1][["support", "residuals"]]
        )

        supports = curves["support"].to_numpy()
        residuals = curves["residuals"].to_numpy()

        for u, e in zip(supports, residuals):

            key = (
                row.scenario,
                row.model_name,
                row.comparison,
                row.regression_model,
                row.ast_kde_samples,
                row.fpc_dimension,
                u
            )

            acc[key][0] += e
            acc[key][1] += e * e
            acc[key][2] += 1

    del df

In [22]:
records = []

for key, (sum_e, sum_e2, n) in acc.items():

    records.append({
        "scenario": key[0],
        "model_name": key[1],
        "comparison": key[2],
        "regression_model": key[3],
        "ast_kde_samples": key[4],
        "fpc_dimension": key[5],
        "support": key[6],
        "bias": sum_e / n,
        "mse": sum_e2 / n
    })

pointwise_metrics = pd.DataFrame(records)

joblib.dump(pointwise_metrics, f"{SIM_MAIN_PATH}cv_metrics.jbl")

['../../data/interim/simulation/20260728/cv_metrics.jbl']

In [23]:
pointwise_metrics = joblib.load(f"{SIM_MAIN_PATH}cv_metrics.jbl")

In [24]:
pointwise_metrics['fpc_dimension'] = (
    pointwise_metrics['fpc_dimension']
    .fillna('-')
)

In [25]:
# # 2. Calculate Pointwise Bias(u) and MSE(u) over the Monte Carlo replications (B)
# # By grouping by 'support' alongside the config, we automatically average over B (n_rep)
# pointwise_metrics = df_flat.groupby(['scenario', 'model_name', 'comparison', 'regression_model', 'fpc_dimension', 'support']).agg(
#     bias=('residuals', 'mean'),                 # 1/B * sum(e_b)
#     mse=('residuals', lambda x: np.mean(x**2))  # 1/B * sum(e_b^2)
# ).reset_index()

# 3. Calculate Supremum and Integral over the grid 'u'
def compute_summary_stats(group):
    # Ensure data is sorted by the grid 'u' before integrating
    group = group.sort_values('support')
    
    u = group['support'].to_numpy()
    bias_u = group['bias'].to_numpy()
    mse_u = group['mse'].to_numpy()
    
    # Supremum (max value across the grid)
    # sup_bias = np.max(np.abs(bias_u))
    sup_mse = np.max(mse_u)
    
    # Integral (Area under the curve using Trapezoidal rule)
    int_bias = np.trapezoid(np.abs(bias_u), u)
    # int_mse = np.trapezoid(mse_u, u)
    
    return pd.Series({
        # 'Supremum_Bias': sup_bias,
        'Integrated_Bias': int_bias,
        'Supremum_MSE': sup_mse
        # 'Integrated_MSE': int_mse
    })

# Collapse the 'support' grid to get your final scalar metrics per model/scenario


final_results = pointwise_metrics.groupby(
    ['scenario', 'model_name', 'comparison', 'regression_model', 'ast_kde_samples', 'fpc_dimension']
).apply(compute_summary_stats).reset_index()

# # display(final_results)

In [26]:
joblib.dump(final_results, f"{SIM_MAIN_PATH}cv_metrics.jbl")

['../../data/interim/simulation/20260728/cv_metrics.jbl']

In [27]:
final_results2 = final_results.copy()
final_results2.loc[:,"regression_model"] = final_results2.loc[:,"regression_model"].str.replace("VAR", "AST")
final_results2.loc[:,"regression_model"] = final_results2.loc[:,"regression_model"].str.replace("adalasso", "Adaptive LASSO")
final_results2.loc[:,"regression_model"] = final_results2.loc[:,"regression_model"].str.replace("randomforest", "Random Forest")
final_results2["fpc_dimension"] = final_results2.fpc_dimension.astype(str).str.replace(r"\.0$", "", regex=True)
final_results2[["scenario_name", "df"]] = final_results2.scenario.str.split("___", expand=True)
final_results2 = final_results2[["scenario", "scenario_name", "df", "comparison", "model_name", 'regression_model', 'ast_kde_samples', 'fpc_dimension', 'Integrated_Bias','Supremum_MSE']]
final_results2.sort_values(by=["scenario", 'comparison', "Integrated_Bias"])

,scenario,scenario_name,df,comparison,model_name,regression_model,ast_kde_samples,fpc_dimension,Integrated_Bias,Supremum_MSE
0,scenario_4___nu=3,scenario_4,nu=3,TfHatFc_Tf,Parametric AST,density,0,3,0.001392,0.024144
2,scenario_4___nu=3,scenario_4,nu=3,TfHatFc_Tf,Parametric AST,kde,1,3,0.114738,9.440641
4,scenario_4___nu=3,scenario_4,nu=3,TfHatFc_Tf,Parametric AST,kde,1000,3,0.116827,9.410991
14,scenario_4___nu=3,scenario_4,nu=3,TfHatFc_Tf,gaussian rot - dynamic,Random Forest,0,3,0.124545,9.388374
15,scenario_4___nu=3,scenario_4,nu=3,TfHatFc_Tf,gaussian rot - dynamic,Random Forest,0,10,0.124572,9.390335
1,scenario_4___nu=3,scenario_4,nu=3,TfHatFc_Tf,Parametric AST,density,0,6,0.124688,1.841936
13,scenario_4___nu=3,scenario_4,nu=3,TfHatFc_Tf,gaussian rot - dynamic,Adaptive LASSO,0,10,0.126357,9.399284
12,scenario_4___nu=3,scenario_4,nu=3,TfHatFc_Tf,gaussian rot - dynamic,Adaptive LASSO,0,3,0.126364,9.400086
3,scenario_4___nu=3,scenario_4,nu=3,TfHatFc_Tf,Parametric AST,kde,1,6,0.179089,10.503741
5,scenario_4___nu=3,scenario_4,nu=3,TfHatFc_Tf,Parametric AST,kde,1000,6,0.179151,10.526655


In [28]:
# 1. Rename the metric columns to match your desired output
df_renamed = final_results2.rename(columns={
    'Integrated_Bias': 'bias',
    'Supremum_MSE': 'mse'
})

# 2. Pivot the dataframe to set rows and columns
report_df = df_renamed.pivot(
    index=['scenario_name', 'df'],
    columns=['fpc_dimension', 'regression_model', 'ast_kde_samples', 'comparison'],
    values=['bias', 'mse']
)

# 3. Reorder the column levels
# pivot puts 'values' (bias, mse) at level 0. fpc_dimension is level 1, regression_model is level 2.
# We want: [fpc_dimension (1), regression_model (2), metrics (0)]
report_df = report_df.reorder_levels([1, 2, 3, 4, 0], axis=1)


# 4. Sort the columns so the hierarchical groupings are displayed cleanly together
report_df = report_df.sort_index(axis=1)
# Move metrics to the top level


# Display the final report
display(report_df)

fpc_dimension                  10                                    \
regression_model   Adaptive LASSO                                     
ast_kde_samples              0                                        
comparison             TfHatFc_Tf           TfHatFc_TfHat             
                             bias       mse          bias       mse   
scenario_name df                                                      
scenario_4    nu=3       0.126357  9.399284      0.009753  1.822311   
              nu=8       0.055252  2.596311      0.008473  1.008575   

fpc_dimension                                                       \
regression_model   Random Forest                                     
ast_kde_samples             0                                        
comparison            TfHatFc_Tf           TfHatFc_TfHat             
                            bias       mse          bias       mse   
scenario_name df                                                     
scenario_4    nu=3      0.124572  9.390335      0.007238  1.761628   
              nu=8      0.052512  2.585907      0.011487  0.977839   

fpc_dimension                   3            ...             6            \
regression_model   Adaptive LASSO            ...       density             
ast_kde_samples              0               ...          0                
comparison             TfHatFc_Tf            ... TfHatFc_TfHat             
                             bias       mse  ...          bias       mse   
scenario_name df                             ...                           
scenario_4    nu=3       0.126364  9.400086  ...      0.109471  2.959997   
              nu=8       0.055265  2.596244  ...      0.084033  3.280600   

fpc_dimension                                                                \
regression_model          kde                                                 
ast_kde_samples          1                                             1000   
comparison         TfHatFc_Tf            TfHatFc_TfHat           TfHatFc_Tf   
                         bias        mse          bias       mse       bias   
scenario_name df                                                              
scenario_4    nu=3   0.179089  10.503741      0.077814  3.685304   0.179151   
              nu=8   0.066344   2.425878      0.022540  2.166424   0.064054   

fpc_dimension                                          
regression_model                                       
ast_kde_samples                                        
comparison                    TfHatFc_TfHat            
                          mse          bias       mse  
scenario_name df                                       
scenario_4    nu=3  10.526655      0.077456  2.474016  
              nu=8   2.407297      0.022364  0.969281  

[2 rows x 40 columns]

In [29]:
# 1. Rename the metric columns to match your desired output
df_renamed = final_results2.rename(columns={
    'Integrated_Bias': 'bias',
    'Supremum_MSE': 'mse'
})

# 2. Pivot the dataframe to set rows and columns
# (Leave values=['bias', 'mse'] here so we can split them cleanly next)
report_df = df_renamed.pivot(
    index=['scenario_name', 'df', 'comparison'],
    columns=['fpc_dimension', 'regression_model', 'ast_kde_samples'],
    values=['bias', 'mse']
)

# 3. Extract into separate tables (this automatically removes the metric level from columns)
bias_df = report_df['bias'].sort_index(axis=1)
mse_df = report_df['mse'].sort_index(axis=1)

# Display the dataframes in your notebook
print("--- Bias Table ---")
display(bias_df)

print("\n--- MSE Table ---")
display(mse_df)

--- Bias Table ---


fpc_dimension                                10                            3  \
regression_model                 Adaptive LASSO Random Forest Adaptive LASSO   
ast_kde_samples                            0             0              0      
scenario_name df   comparison                                                  
scenario_4    nu=3 TfHatFc_Tf          0.126357      0.124572       0.126364   
                   TfHatFc_TfHat       0.009753      0.007238       0.009748   
              nu=8 TfHatFc_Tf          0.055252      0.052512       0.055265   
                   TfHatFc_TfHat       0.008473      0.011487       0.008454   

fpc_dimension                                                                 \
regression_model                 Random Forest   density       kde             
ast_kde_samples                           0         0         1         1000   
scenario_name df   comparison                                                  
scenario_4    nu=3 TfHatFc_Tf         0.124545  0.001392  0.114738  0.116827   
                   TfHatFc_TfHat      0.007290  0.121121  0.005833  0.004624   
              nu=8 TfHatFc_Tf         0.052656  0.141074  0.141814  0.143092   
                   TfHatFc_TfHat      0.011348  0.179961  0.097125  0.098175   

fpc_dimension                            6                      
regression_model                   density       kde            
ast_kde_samples                       0         1         1000  
scenario_name df   comparison                                   
scenario_4    nu=3 TfHatFc_Tf     0.124688  0.179089  0.179151  
                   TfHatFc_TfHat  0.109471  0.077814  0.077456  
              nu=8 TfHatFc_Tf     0.030601  0.066344  0.064054  
                   TfHatFc_TfHat  0.084033  0.022540  0.022364


--- MSE Table ---


fpc_dimension                                10                            3  \
regression_model                 Adaptive LASSO Random Forest Adaptive LASSO   
ast_kde_samples                            0             0              0      
scenario_name df   comparison                                                  
scenario_4    nu=3 TfHatFc_Tf          9.399284      9.390335       9.400086   
                   TfHatFc_TfHat       1.822311      1.761628       1.823311   
              nu=8 TfHatFc_Tf          2.596311      2.585907       2.596244   
                   TfHatFc_TfHat       1.008575      0.977839       1.006389   

fpc_dimension                                                                 \
regression_model                 Random Forest   density       kde             
ast_kde_samples                           0         0         1         1000   
scenario_name df   comparison                                                  
scenario_4    nu=3 TfHatFc_Tf         9.388374  0.024144  9.440641  9.410991   
                   TfHatFc_TfHat      1.759318  9.024843  3.401990  1.653576   
              nu=8 TfHatFc_Tf         2.584666  0.810425  2.163759  2.029948   
                   TfHatFc_TfHat      0.973253  3.756935  2.956547  1.597520   

fpc_dimension                            6                        
regression_model                   density        kde             
ast_kde_samples                       0          1          1000  
scenario_name df   comparison                                     
scenario_4    nu=3 TfHatFc_Tf     1.841936  10.503741  10.526655  
                   TfHatFc_TfHat  2.959997   3.685304   2.474016  
              nu=8 TfHatFc_Tf     0.110099   2.425878   2.407297  
                   TfHatFc_TfHat  3.280600   2.166424   0.969281

In [35]:
bias_df.T.sort_values(by="regression_model")

scenario_name                                  scenario_4                \
df                                                   nu=3                 
comparison                                     TfHatFc_Tf TfHatFc_TfHat   
fpc_dimension regression_model ast_kde_samples                            
10            Adaptive LASSO   0                 0.126357      0.009753   
3             Adaptive LASSO   0                 0.126364      0.009748   
10            Random Forest    0                 0.124572      0.007238   
3             Random Forest    0                 0.124545      0.007290   
              density          0                 0.001392      0.121121   
6             density          0                 0.124688      0.109471   
3             kde              1                 0.114738      0.005833   
                               1000              0.116827      0.004624   
6             kde              1                 0.179089      0.077814   
                               1000              0.179151      0.077456   

scenario_name                                                            
df                                                   nu=8                
comparison                                     TfHatFc_Tf TfHatFc_TfHat  
fpc_dimension regression_model ast_kde_samples                           
10            Adaptive LASSO   0                 0.055252      0.008473  
3             Adaptive LASSO   0                 0.055265      0.008454  
10            Random Forest    0                 0.052512      0.011487  
3             Random Forest    0                 0.052656      0.011348  
              density          0                 0.141074      0.179961  
6             density          0                 0.030601      0.084033  
3             kde              1                 0.141814      0.097125  
                               1000              0.143092      0.098175  
6             kde              1                 0.066344      0.022540  
                               1000              0.064054      0.022364

In [37]:
import matplotlib.colors as mcolors

# 1. Define soft, desaturated pastel hex codes
# Low values (Best) = Soft Green, Mid = Soft Yellow, High values (Worst) = Soft Red
# pastel_colors = ["#3066be", "#c3c3c3", "#ff9505"]
pastel_colors = ["#60d394", "#c3c3c3", "#ee6055"]

# 2. Create the custom colormap
soft_cmap = mcolors.LinearSegmentedColormap.from_list("soft_rdylgn", pastel_colors)

# 3. Apply it to your dataframes exactly like before
styled_bias = bias_df.T.sort_values(by="regression_model").style.background_gradient(cmap=soft_cmap, axis=0).format("{:.4f}")
styled_mse = mse_df.T.sort_values(by="regression_model").style.background_gradient(cmap=soft_cmap, axis=0).format("{:.4f}")

# Display to see the much lighter, cleaner look
display(styled_bias)
display(styled_mse)

In [ ]:
styled_bias.sort